# 4. Speech Pipeline — Whisper Transcription & Sentence Embeddings

Two independent, resumable stages over the **speech-labeled** subset of `audio_speech_labels.csv` (Notebook 1's output). Neither stage re-downloads anything — both read straight from Notebook 1's local `data/audio/` cache.

1. **Transcribe** — run [`faster-whisper`](https://github.com/SYSTRAN/faster-whisper) (`large-v3` by default) on every `is_speech == True` clip's cached audio, and append the transcript (plus detected language, duration, segment count) to a new `data/audio_transcriptions.csv`. `audio_speech_labels.csv` itself is never modified.
2. **Embed** — run a `sentence-transformers` model (`all-mpnet-base-v2` by default — the same model family used for the podcast recommender's text embeddings, for cross-project consistency) over the completed, non-empty transcripts, and write normalized embedding vectors to `data/audio_transcript_embeddings.csv`.

**Device notes:**
- `faster-whisper` runs on CTranslate2, which supports **CUDA or CPU only** — there's no Apple Metal/MPS backend, so on a Mac this stage falls back to CPU (`int8` compute type) and will be slow for `large-v3`. On a CUDA box it auto-selects `float16`. Override via `.env` (`WHISPER_DEVICE`, `WHISPER_COMPUTE_TYPE`) if needed — e.g. on Quest/HPC with a GPU.
- `sentence-transformers` is plain PyTorch, so it *does* get Apple MPS acceleration locally, and CUDA wherever available.

**Dependencies not yet in `requirements.txt`:** `faster-whisper`, `sentence-transformers`. Install them with the cell below (mirrors how `YAMNet_Top5.ipynb` installs `tensorflow` ad hoc).

## 0. Install dependencies

Run once in the same `.venv` used for the other notebooks. Restart the kernel if VS Code asks you to, then continue from the next cell.

In [2]:
%pip install -U faster-whisper sentence-transformers


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import subprocess
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

load_dotenv()

/Users/prakeerthprasad/Desktop/FORGE - Says/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Configuration

All values come from `.env` at the repo root, with sensible fallbacks. Device/compute-type are auto-detected unless explicitly overridden.

In [4]:
# Locations (relative to this notebook's directory, i.e. jupyter_notebooks/)
DATA_DIR = Path(os.getenv("DATA_DIR", "../data"))
AUDIO_DIR = DATA_DIR / "audio"

# Input: Notebook 1's output. Only rows with is_speech == True are processed.
SPEECH_LABELS_CSV = DATA_DIR / os.getenv("SPEECH_LABELS_CSV", "audio_speech_labels.csv")

# Outputs: brand-new CSVs. Neither SPEECH_LABELS_CSV nor audio_download_log.csv
# is ever modified by this notebook.
TRANSCRIPTS_CSV = DATA_DIR / os.getenv("TRANSCRIPTS_CSV", "audio_transcriptions.csv")
TRANSCRIPT_EMBEDDINGS_CSV = DATA_DIR / os.getenv("TRANSCRIPT_EMBEDDINGS_CSV", "audio_transcript_embeddings.csv")

ID_COLUMN = os.getenv("ID_COLUMN", "id")
AUDIO_URL_COLUMN = os.getenv("AUDIO_URL_COLUMN", "streamableUrl")
SPEECH_COLUMN = "is_speech"

SAMPLE_RATE = 16_000  # both faster-whisper and the cached-audio decoder expect 16kHz mono.

# Save progress every N newly processed rows in each stage.
SAVE_EVERY = int(os.getenv("SAVE_EVERY", "10"))

# Maximum time allowed for ffmpeg to decode one local cached file.
FFMPEG_TIMEOUT_SECONDS = int(os.getenv("FFMPEG_TIMEOUT_SECONDS", "120"))

SUPPORTED_AUDIO_SUFFIXES = {".mp3", ".m4a", ".wav", ".ogg", ".aac", ".flac"}


# --- Whisper configuration ---

WHISPER_MODEL_SIZE = os.getenv("WHISPER_MODEL_SIZE", "large-v3")

# Leave unset to let Whisper auto-detect the spoken language.
_language_env = os.getenv("WHISPER_LANGUAGE", "").strip()
WHISPER_LANGUAGE = _language_env if _language_env else None

# Parallel inference streams inside the single, shared WhisperModel --
# what faster-whisper/CTranslate2 uses to safely serve transcribe() calls
# from multiple Python threads at once. Independent of MAX_WORKERS below.
WHISPER_NUM_WORKERS = int(os.getenv("WHISPER_NUM_WORKERS", "4"))

# Threads CTranslate2 uses per inference stream when running on CPU.
# 0 lets CTranslate2 pick automatically. Ignored on cuda.
WHISPER_CPU_THREADS = int(os.getenv("WHISPER_CPU_THREADS", "0"))

# Rows processed concurrently. Each one decodes cached audio via ffmpeg
# (CPU-bound) and then calls the shared Whisper model above. Keep this
# >= WHISPER_NUM_WORKERS so the model always has work queued.
MAX_WORKERS = int(os.getenv("TRANSCRIBE_MAX_WORKERS", "8"))


def pick_whisper_device_and_compute_type() -> tuple[str, str]:
    """CTranslate2 (faster-whisper's backend) only supports CUDA or CPU --
    there is no Apple MPS/Metal support, unlike plain PyTorch models."""
    if torch.cuda.is_available():
        return "cuda", "float16"
    return "cpu", "int8"


_auto_whisper_device, _auto_whisper_compute_type = pick_whisper_device_and_compute_type()
WHISPER_DEVICE = os.getenv("WHISPER_DEVICE", "").strip() or _auto_whisper_device
WHISPER_COMPUTE_TYPE = os.getenv("WHISPER_COMPUTE_TYPE", "").strip() or _auto_whisper_compute_type


# --- Sentence embedding configuration ---

# Same model family as the podcast recommender's text embeddings, for
# consistency if the two projects are ever compared or merged.
TEXT_MODEL_NAME = os.getenv("TEXT_EMBEDDING_MODEL", "all-mpnet-base-v2")
TEXT_EMBEDDING_BATCH_SIZE = int(os.getenv("TEXT_EMBEDDING_BATCH_SIZE", "64"))


def pick_text_embedding_device() -> str:
    """Plain PyTorch model, so CUDA and Apple MPS both work, unlike Whisper."""
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


TEXT_EMBEDDING_DEVICE = os.getenv("TEXT_EMBEDDING_DEVICE", "").strip() or pick_text_embedding_device()

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Whisper model:   {WHISPER_MODEL_SIZE} ({WHISPER_DEVICE}, {WHISPER_COMPUTE_TYPE})")
print(f"Embedding model: {TEXT_MODEL_NAME} ({TEXT_EMBEDDING_DEVICE})")

Whisper model:   large-v3 (cpu, int8)
Embedding model: all-mpnet-base-v2 (mps)


## Locate cached audio

Reuses Notebook 1's exact cache-path rule (extension derived from the streamable URL), with a fallback scan by ID for anything that doesn't match. No audio is downloaded here — everything is read from `AUDIO_DIR`.

In [5]:
def cache_path_for(row_id: str, url: str) -> Path:
    """Same cache-path rule used by Notebook 1."""
    suffix = Path(urlparse(url).path).suffix.lower() or ".mp3"
    if suffix not in SUPPORTED_AUDIO_SUFFIXES:
        suffix = ".mp3"
    return AUDIO_DIR / f"{row_id}{suffix}"


def find_cached_audio(row: dict) -> Path | None:
    """Find the audio downloaded by Notebook 1, or None if it isn't cached."""
    row_id = str(row.get(ID_COLUMN)).strip()
    url_value = row.get(AUDIO_URL_COLUMN)

    if pd.notna(url_value) and str(url_value).strip():
        expected_path = cache_path_for(row_id, str(url_value).strip())
        if expected_path.exists() and expected_path.stat().st_size > 0:
            return expected_path

    fallback_candidates = [
        candidate
        for suffix in SUPPORTED_AUDIO_SUFFIXES
        if (candidate := AUDIO_DIR / f"{row_id}{suffix}").exists() and candidate.stat().st_size > 0
    ]

    if not fallback_candidates:
        return None

    # Prefer the largest complete cached file if multiple formats exist.
    return max(fallback_candidates, key=lambda path: path.stat().st_size)


def decode_audio_for_whisper(audio_path: Path, sample_rate: int = SAMPLE_RATE) -> np.ndarray:
    """Decode a local audio file into mono float32 PCM samples at 16kHz."""
    command = [
        "ffmpeg",
        "-nostdin",
        "-hide_banner",
        "-loglevel", "error",
        "-i", str(audio_path),
        "-vn",
        "-ac", "1",
        "-ar", str(sample_rate),
        "-f", "f32le",
        "-",
    ]

    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=FFMPEG_TIMEOUT_SECONDS,
        )
    except subprocess.TimeoutExpired as error:
        raise TimeoutError(f"ffmpeg timed out while decoding {audio_path.name}") from error

    if result.returncode != 0:
        error_message = result.stderr.decode("utf-8", errors="ignore").strip()
        raise RuntimeError(error_message or f"ffmpeg failed to decode {audio_path.name}")

    waveform = np.frombuffer(result.stdout, dtype=np.float32)
    waveform = np.nan_to_num(waveform, nan=0.0, posinf=0.0, neginf=0.0)

    if waveform.size == 0:
        raise ValueError(f"Decoded waveform is empty: {audio_path.name}")

    return waveform

## Stage 1 — Transcribe speech clips with `faster-whisper`

A single `WhisperModel` instance is built once (before any worker threads start) and shared across all of them — faster-whisper/CTranslate2 is explicitly designed to be called concurrently this way via `num_workers`, so this gets parallelism without paying for N separate model copies in memory. `MAX_WORKERS` mostly controls how many ffmpeg decodes run at once so the model always has queued work.

In [6]:
def build_whisper_model() -> WhisperModel:
    return WhisperModel(
        WHISPER_MODEL_SIZE,
        device=WHISPER_DEVICE,
        compute_type=WHISPER_COMPUTE_TYPE,
        cpu_threads=WHISPER_CPU_THREADS,
        num_workers=WHISPER_NUM_WORKERS,
    )


def transcribe_audio(whisper_model: WhisperModel, audio_path: Path) -> dict:
    waveform = decode_audio_for_whisper(audio_path)

    segments, info = whisper_model.transcribe(
        waveform,
        language=WHISPER_LANGUAGE,
    )

    segments = list(segments)
    transcript = " ".join(segment.text.strip() for segment in segments).strip()

    return {
        "transcript": transcript,
        "language": info.language,
        "language_probability": info.language_probability,
        "audio_duration_seconds": info.duration,
        "segment_count": len(segments),
    }


def transcribe_row(whisper_model: WhisperModel, row: dict) -> dict:
    """Transcribe one row's cached audio. Returns the DB row + transcript fields."""
    result = dict(row)
    result.update({
        "transcript": None,
        "language": None,
        "language_probability": None,
        "audio_duration_seconds": None,
        "segment_count": None,
        "transcription_status": None,
        "transcription_error": None,
    })

    row_id = row.get(ID_COLUMN)

    try:
        audio_path = find_cached_audio(row)
        if audio_path is None:
            raise FileNotFoundError("Audio not downloaded (see Notebook 1, Stage 2).")

        result.update(transcribe_audio(whisper_model, audio_path))
        result["transcription_status"] = "completed"

    except Exception as error:
        result["transcription_status"] = "failed"
        result["transcription_error"] = str(error)
        print(f"[TRANSCRIBE FAILED] {ID_COLUMN}={row_id}: {error}")

    return result

In [7]:
def load_existing_results(output_csv: Path) -> pd.DataFrame:
    if not output_csv.exists():
        return pd.DataFrame()

    try:
        existing = pd.read_csv(output_csv)
        if ID_COLUMN in existing.columns:
            existing[ID_COLUMN] = existing[ID_COLUMN].astype(str)
        return existing
    except Exception as error:
        print(f"Could not read existing CSV {output_csv}: {error}")
        return pd.DataFrame()


def get_processed_ids(existing_results: pd.DataFrame) -> set:
    if existing_results.empty or ID_COLUMN not in existing_results.columns:
        return set()
    return set(existing_results[ID_COLUMN].astype(str).tolist())


def save_results(existing_results: pd.DataFrame, new_results: list, output_csv: Path) -> pd.DataFrame:
    new_dataframe = pd.DataFrame(new_results)

    if existing_results.empty:
        combined_dataframe = new_dataframe
    else:
        combined_dataframe = pd.concat([existing_results, new_dataframe], ignore_index=True)

    combined_dataframe.to_csv(output_csv, index=False)
    return combined_dataframe

In [ ]:
if not SPEECH_LABELS_CSV.exists():
    raise FileNotFoundError(
        f"Could not find: {SPEECH_LABELS_CSV}\n"
        "Run Notebook 1 through Stage 3 first so that audio_speech_labels.csv exists."
    )

speech_labels = pd.read_csv(SPEECH_LABELS_CSV)
speech_labels[ID_COLUMN] = speech_labels[ID_COLUMN].astype(str)

if SPEECH_COLUMN not in speech_labels.columns:
    raise KeyError(f"Expected Notebook 1's '{SPEECH_COLUMN}' column, but it was not found.")

speech_rows = speech_labels[speech_labels[SPEECH_COLUMN] == True]

existing_transcripts = load_existing_results(TRANSCRIPTS_CSV)
processed_ids = get_processed_ids(existing_transcripts)

rows_to_transcribe = speech_rows[~speech_rows[ID_COLUMN].isin(processed_ids)]

print(f"Speech-labeled rows:   {len(speech_rows)}")
print(f"Already transcribed:   {len(processed_ids)}")
print(f"Remaining rows:        {len(rows_to_transcribe)}")
print(f"Download workers:      {MAX_WORKERS}")
print(f"Inference workers:     {WHISPER_NUM_WORKERS}")

whisper_model = build_whisper_model()

new_transcripts = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(transcribe_row, whisper_model, row.to_dict())
        for _, row in rows_to_transcribe.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Transcribing audio", unit="clip"):
        new_transcripts.append(future.result())

        if len(new_transcripts) % SAVE_EVERY == 0:
            existing_transcripts = save_results(existing_transcripts, new_transcripts, TRANSCRIPTS_CSV)
            new_transcripts = []

if new_transcripts:
    existing_transcripts = save_results(existing_transcripts, new_transcripts, TRANSCRIPTS_CSV)

print("\nTranscription stage finished.")
print(f"Results saved to: {TRANSCRIPTS_CSV}")

Speech-labeled rows:   4463
Already transcribed:   510
Remaining rows:        3953
Download workers:      8
Inference workers:     4


Transcribing audio:   0%|          | 18/3953 [05:16<19:14:48, 17.61s/clip]


## Stage 2 — Embed transcripts into a shared embedding space

Runs over completed, non-empty transcripts from Stage 1. Embeddings are L2-normalized so cosine similarity reduces to a dot product, and are stored as comma-separated strings (matching the podcast recommender's CSV embedding format) so they round-trip cleanly through `pandas`/`csv` without needing a binary format.

In [ ]:
def encode_vector(vector: np.ndarray) -> str:
    return ",".join(f"{value:.6f}" for value in vector)


def decode_vector(text: str) -> np.ndarray:
    return np.array(text.split(","), dtype=np.float32)

In [ ]:
transcripts = pd.read_csv(TRANSCRIPTS_CSV)
transcripts[ID_COLUMN] = transcripts[ID_COLUMN].astype(str)

if "transcription_status" in transcripts.columns:
    transcripts = transcripts[transcripts["transcription_status"] == "completed"]

transcripts["transcript"] = transcripts["transcript"].fillna("").astype(str).str.strip()
transcripts = transcripts[transcripts["transcript"] != ""]

existing_embeddings = load_existing_results(TRANSCRIPT_EMBEDDINGS_CSV)
embedded_ids = get_processed_ids(existing_embeddings)

rows_to_embed = transcripts[~transcripts[ID_COLUMN].isin(embedded_ids)]

print(f"Completed non-empty transcripts: {len(transcripts)}")
print(f"Already embedded:                {len(embedded_ids)}")
print(f"Remaining rows:                  {len(rows_to_embed)}")

if rows_to_embed.empty:
    print("Nothing new to embed.")
else:
    embedding_model = SentenceTransformer(TEXT_MODEL_NAME, device=TEXT_EMBEDDING_DEVICE)

    embeddings = embedding_model.encode(
        rows_to_embed["transcript"].tolist(),
        batch_size=TEXT_EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    new_embeddings = [
        {
            ID_COLUMN: row_id,
            "transcript_embedding": encode_vector(vector),
        }
        for row_id, vector in zip(rows_to_embed[ID_COLUMN].values, embeddings)
    ]

    existing_embeddings = save_results(existing_embeddings, new_embeddings, TRANSCRIPT_EMBEDDINGS_CSV)

    print("\nEmbedding stage finished.")
    print(f"Results saved to: {TRANSCRIPT_EMBEDDINGS_CSV}")

## Verify

In [ ]:
result_transcripts = pd.read_csv(TRANSCRIPTS_CSV) if TRANSCRIPTS_CSV.exists() else pd.DataFrame()
result_embeddings = pd.read_csv(TRANSCRIPT_EMBEDDINGS_CSV) if TRANSCRIPT_EMBEDDINGS_CSV.exists() else pd.DataFrame()

if not result_transcripts.empty:
    print("Transcription status:")
    print(result_transcripts["transcription_status"].value_counts(dropna=False))
    print()
    display(result_transcripts[[ID_COLUMN, "transcription_status", "language", "segment_count", "transcript"]].head(10))

if not result_embeddings.empty:
    sample_dim = len(decode_vector(result_embeddings["transcript_embedding"].iloc[0]))
    print(f"\nEmbedded rows: {len(result_embeddings)}  |  Embedding dimension: {sample_dim}")

## Explore Speech Recommendations

Pick a few random speech clips and find their most similar matches based on transcript embeddings (cosine similarity via dot product, since vectors are L2-normalized).

In [ ]:
import glob
import random
from IPython.display import display, Audio

transcripts_df = pd.read_csv(TRANSCRIPTS_CSV)
transcripts_df[ID_COLUMN] = transcripts_df[ID_COLUMN].astype(str)

embeddings_df = pd.read_csv(TRANSCRIPT_EMBEDDINGS_CSV)
embeddings_df[ID_COLUMN] = embeddings_df[ID_COLUMN].astype(str)

merged = transcripts_df.merge(embeddings_df, on=ID_COLUMN, how="inner")
merged = merged[merged["transcription_status"] == "completed"]
merged = merged[merged["transcript"].fillna("").str.strip() != ""]

embedding_matrix = np.vstack(merged["transcript_embedding"].apply(decode_vector).values)

audio_files = glob.glob(os.path.join(str(AUDIO_DIR), "*.*"))
audio_path_map = {os.path.splitext(os.path.basename(f))[0]: f for f in audio_files}

print(f"Speech clips with embeddings: {len(merged)}")
print(f"Embedding dimension:          {embedding_matrix.shape[1]}")
print(f"Audio files found:            {len(audio_path_map)}")

In [ ]:
NUM_QUERIES = 3
TOP_K = 5

random.seed(42)
query_indices = random.sample(range(len(merged)), NUM_QUERIES)

for qi in query_indices:
    query_id = merged.iloc[qi][ID_COLUMN]
    query_transcript = merged.iloc[qi]["transcript"]
    query_lang = merged.iloc[qi].get("language", "?")

    query_vec = embedding_matrix[qi].reshape(1, -1)
    scores = (embedding_matrix @ query_vec.T).flatten()

    # exclude the query itself
    scores[qi] = -1.0
    top_indices = np.argsort(scores)[::-1][:TOP_K]

    print(f"\n{'='*70}")
    print(f"QUERY  id={query_id}  lang={query_lang}")
    print(f"Transcript: {query_transcript}")

    q_path = audio_path_map.get(query_id)
    if q_path:
        display(Audio(q_path))

    print(f"\nTop {TOP_K} similar speech clips:")
    for rank, idx in enumerate(top_indices, 1):
        match_id = merged.iloc[idx][ID_COLUMN]
        match_transcript = merged.iloc[idx]["transcript"]
        match_lang = merged.iloc[idx].get("language", "?")
        score = scores[idx]

        print(f"\n  {rank}. id={match_id}  score={score:.4f}  lang={match_lang}")
        print(f"     {match_transcript}")

        m_path = audio_path_map.get(match_id)
        if m_path:
            display(Audio(m_path))

In [ ]:
CUSTOM_QUERY_ID = ""  # <-- paste an id here to query a specific clip

if CUSTOM_QUERY_ID.strip():
    match = merged[merged[ID_COLUMN] == CUSTOM_QUERY_ID.strip()]
    if match.empty:
        print(f"ID '{CUSTOM_QUERY_ID}' not found in embedded speech clips.")
    else:
        idx = match.index[0]
        pos = merged.index.get_loc(idx)
        query_vec = embedding_matrix[pos].reshape(1, -1)
        scores = (embedding_matrix @ query_vec.T).flatten()
        scores[pos] = -1.0
        top_indices = np.argsort(scores)[::-1][:TOP_K]

        print(f"QUERY  id={CUSTOM_QUERY_ID}")
        print(f"Transcript: {match.iloc[0]['transcript']}")
        q_path = audio_path_map.get(CUSTOM_QUERY_ID.strip())
        if q_path:
            display(Audio(q_path))

        print(f"\nTop {TOP_K} similar speech clips:")
        for rank, ti in enumerate(top_indices, 1):
            m_id = merged.iloc[ti][ID_COLUMN]
            m_transcript = merged.iloc[ti]["transcript"]
            score = scores[ti]
            print(f"\n  {rank}. id={m_id}  score={score:.4f}")
            print(f"     {m_transcript}")
            m_path = audio_path_map.get(m_id)
            if m_path:
                display(Audio(m_path))
else:
    print("Set CUSTOM_QUERY_ID above to query a specific clip.")